# Kaggle Playground S6E4 — Predicting Irrigation Need

**Competition:** https://www.kaggle.com/competitions/playground-series-s6e4  
**Task:** Multi-class classification (Low / Medium / High)  
**Metric:** Balanced Accuracy  
**Deadline:** May 3, 2026  

### Game Plan
1. Setup & Data Loading
2. Exploratory Data Analysis (EDA)
3. Preprocessing
4. Baseline Models (Logistic Regression, Random Forest)
5. Gradient Boosting (LightGBM, XGBoost, CatBoost)
6. Feature Engineering
7. Hyperparameter Tuning (Optuna)
8. Ensembling
9. Final Submission

---
## 1. Setup & Imports

In [ ]:
# Install dependencies (run this cell once, then you can skip it on future runs)
!pip install lightgbm xgboost catboost optuna scikit-learn pandas numpy matplotlib seaborn --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.metrics import balanced_accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
N_FOLDS = 5
np.random.seed(SEED)

print('All libraries loaded successfully!')

---
## 2. Load Data

In [ ]:
# Update these paths if needed
DATA_DIR = 'TrainTestData/'  # folder containing train.csv, test.csv, sample_submission.csv

train = pd.read_csv(DATA_DIR + 'train.csv')
test  = pd.read_csv(DATA_DIR + 'test.csv')
sub   = pd.read_csv(DATA_DIR + 'sample_submission.csv')

print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')
print(f'Sample submission shape: {sub.shape}')

In [ ]:
train.head()

In [ ]:
train.dtypes

---
## 3. Exploratory Data Analysis (EDA)

In [ ]:
# --- Missing Values ---
missing = train.isnull().sum()
print('Missing values in train:')
print(missing[missing > 0] if missing.any() else 'None!')

In [ ]:
# --- Target Distribution ---
target_counts = train['Irrigation_Need'].value_counts()
print('Target class distribution:')
print(target_counts)
print(f'\nClass balance (%):')
print((target_counts / len(train) * 100).round(2))

fig, ax = plt.subplots(figsize=(6, 4))
target_counts.plot(kind='bar', color=['steelblue', 'coral', 'seagreen'], ax=ax)
ax.set_title('Target Class Distribution (Irrigation_Need)', fontsize=13)
ax.set_xlabel('Class')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# --- Identify feature types ---
TARGET = 'Irrigation_Need'
ID_COL = 'id'

cat_cols = [c for c in train.columns if train[c].dtype == 'object' and c != TARGET]
num_cols = [c for c in train.columns if train[c].dtype != 'object' and c not in [ID_COL, TARGET]]

print(f'Categorical features ({len(cat_cols)}): {cat_cols}')
print(f'Numeric features    ({len(num_cols)}): {num_cols}')

In [ ]:
# --- Categorical feature cardinality ---
print('Unique values per categorical feature:')
for col in cat_cols:
    print(f'  {col}: {train[col].nunique()} unique -> {train[col].unique()[:8]}')

In [ ]:
# --- Numeric feature distributions ---
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(train[col].dropna(), bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('')

# Hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numeric Feature Distributions', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# --- Correlation heatmap (numeric features) ---
corr = train[num_cols].corr()

plt.figure(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, annot_kws={'size': 8})
plt.title('Correlation Matrix — Numeric Features', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# --- Feature vs Target (boxplots for top numeric features) ---
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols[:8]):
    train.boxplot(column=col, by=TARGET, ax=axes[i])
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('')

plt.suptitle('Numeric Features by Irrigation_Need Class', fontsize=13)
plt.tight_layout()
plt.show()

---
## 4. Preprocessing

In [ ]:
# --- Encode target ---
label_map = {'Low': 0, 'Medium': 1, 'High': 2}
inv_label_map = {v: k for k, v in label_map.items()}

y = train[TARGET].map(label_map).values
print('Target encoded. Class mapping:', label_map)
print('Unique values in y:', np.unique(y))

In [ ]:
# --- Combine train/test for consistent encoding ---
all_data = pd.concat([train.drop(columns=[TARGET]), test], axis=0, ignore_index=True)

# Label-encode categorical columns
le = LabelEncoder()
for col in cat_cols:
    all_data[col] = le.fit_transform(all_data[col].astype(str))

# Split back
n_train = len(train)
train_enc = all_data.iloc[:n_train].copy()
test_enc  = all_data.iloc[n_train:].copy()

feature_cols = [c for c in train_enc.columns if c != ID_COL]
X = train_enc[feature_cols].values
X_test = test_enc[feature_cols].values

print(f'X shape: {X.shape}, X_test shape: {X_test.shape}')

---
## 5. Baseline Models

Quick sanity-check with simple models to get on the leaderboard fast.

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

def cv_balanced_accuracy(model, X, y, skf):
    """Run stratified k-fold CV and return mean balanced accuracy."""
    scores = []
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)
        score = balanced_accuracy_score(y_val, preds)
        scores.append(score)
        print(f'  Fold {fold+1}: {score:.4f}')
    mean_score = np.mean(scores)
    print(f'  Mean Balanced Accuracy: {mean_score:.4f} ± {np.std(scores):.4f}')
    return scores

In [ ]:
# --- Logistic Regression baseline ---
print('=== Logistic Regression Baseline ===')
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

lr = LogisticRegression(max_iter=500, random_state=SEED, class_weight='balanced')
lr_scores = cv_balanced_accuracy(lr, X_scaled, y, skf)

In [ ]:
# --- Random Forest baseline ---
print('=== Random Forest Baseline ===')
rf = RandomForestClassifier(n_estimators=100, random_state=SEED, class_weight='balanced', n_jobs=-1)
rf_scores = cv_balanced_accuracy(rf, X, y, skf)

---
## 6. Gradient Boosting Models

LightGBM, XGBoost, and CatBoost — these are the workhorses for tabular competitions.

In [ ]:
# Helper: CV with out-of-fold predictions (needed for ensembling later)
def cv_with_oof(model_fn, X, y, X_test, skf, model_name='Model'):
    """
    Run cross-validation, store OOF predictions and test predictions.
    Returns: oof_preds (soft), test_preds_avg (soft), scores
    """
    n_classes = len(np.unique(y))
    oof_preds = np.zeros((len(y), n_classes))
    test_preds = np.zeros((len(X_test), n_classes))
    scores = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        model = model_fn()
        model.fit(X_tr, y_tr)

        oof_preds[val_idx] = model.predict_proba(X_val)
        test_preds += model.predict_proba(X_test) / N_FOLDS

        fold_preds = np.argmax(oof_preds[val_idx], axis=1)
        score = balanced_accuracy_score(y_val, fold_preds)
        scores.append(score)
        print(f'  Fold {fold+1}: {score:.4f}')

    oof_score = balanced_accuracy_score(y, np.argmax(oof_preds, axis=1))
    print(f'  [{model_name}] OOF Balanced Accuracy: {oof_score:.4f} ± {np.std(scores):.4f}')
    return oof_preds, test_preds, scores, oof_score

In [ ]:
# --- LightGBM ---
print('=== LightGBM ===')

LGBM_PARAMS = dict(
    objective='multiclass',
    num_class=3,
    metric='multi_logloss',
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight='balanced',
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1,
)

def make_lgbm():
    return lgb.LGBMClassifier(**LGBM_PARAMS)

lgbm_oof, lgbm_test, lgbm_scores, lgbm_oof_score = cv_with_oof(make_lgbm, X, y, X_test, skf, 'LightGBM')

In [ ]:
# --- Feature Importance (LightGBM) ---
lgbm_final = lgb.LGBMClassifier(**LGBM_PARAMS)
lgbm_final.fit(X, y)

feat_imp = pd.DataFrame({
    'feature': feature_cols,
    'importance': lgbm_final.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=feat_imp, x='importance', y='feature', palette='viridis')
plt.title('LightGBM Feature Importance', fontsize=13)
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print(feat_imp.to_string(index=False))

In [ ]:
# --- XGBoost ---
print('=== XGBoost ===')

XGB_PARAMS = dict(
    objective='multi:softprob',
    num_class=3,
    eval_metric='mlogloss',
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',  # fast on large datasets
    random_state=SEED,
    n_jobs=-1,
    verbosity=0,
)

def make_xgb():
    return xgb.XGBClassifier(**XGB_PARAMS)

xgb_oof, xgb_test, xgb_scores, xgb_oof_score = cv_with_oof(make_xgb, X, y, X_test, skf, 'XGBoost')

In [ ]:
# --- CatBoost ---
print('=== CatBoost ===')

# CatBoost can handle categoricals natively — pass cat feature indices
cat_feature_indices = [feature_cols.index(c) for c in cat_cols if c in feature_cols]
print(f'Categorical feature indices for CatBoost: {cat_feature_indices}')

CBC_PARAMS = dict(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function='MultiClass',
    eval_metric='Accuracy',
    random_seed=SEED,
    verbose=0,
    auto_class_weights='Balanced',
    cat_features=cat_feature_indices,
)

def make_cbc():
    return CatBoostClassifier(**CBC_PARAMS)

cbc_oof, cbc_test, cbc_scores, cbc_oof_score = cv_with_oof(make_cbc, X, y, X_test, skf, 'CatBoost')

In [ ]:
# --- Model Comparison Summary ---
results = pd.DataFrame({
    'Model': ['LightGBM', 'XGBoost', 'CatBoost'],
    'OOF_Balanced_Accuracy': [lgbm_oof_score, xgb_oof_score, cbc_oof_score]
}).sort_values('OOF_Balanced_Accuracy', ascending=False)

print('\n=== Model Comparison ===')
print(results.to_string(index=False))

---
## 7. Feature Engineering

Add interaction terms and domain-inspired features to give models more signal.

In [ ]:
def engineer_features(df):
    df = df.copy()

    # --- Interaction features ---
    # High rainfall + high humidity → likely low irrigation need
    df['Rain_x_Humidity'] = df['Rainfall_mm'] * df['Humidity']

    # High temperature + low rainfall → high irrigation need
    df['Temp_x_Rain_inv'] = df['Temperature_C'] / (df['Rainfall_mm'] + 1)

    # Soil moisture vs previous irrigation
    df['Moisture_prev_ratio'] = df['Soil_Moisture'] / (df['Previous_Irrigation_mm'] + 1)

    # Water deficit proxy
    df['Water_deficit'] = df['Temperature_C'] * df['Sunlight_Hours'] - df['Rainfall_mm'] - df['Soil_Moisture']

    # Evapotranspiration proxy (simplified Hargreaves-style)
    df['ET_proxy'] = 0.0023 * (df['Temperature_C'] + 17.8) * df['Sunlight_Hours'] ** 0.5

    # Sunlight × temperature stress
    df['Sun_Temp'] = df['Sunlight_Hours'] * df['Temperature_C']

    # Field area × soil moisture (total water in field)
    df['Total_soil_moisture'] = df['Field_Area_hectare'] * df['Soil_Moisture']

    # Wind drying effect
    df['Wind_x_Temp'] = df['Wind_Speed_kmh'] * df['Temperature_C']

    return df


# Apply to combined train/test (re-encode categoricals)
train_fe = engineer_features(train.drop(columns=[TARGET]))
test_fe  = engineer_features(test)

all_data_fe = pd.concat([train_fe, test_fe], axis=0, ignore_index=True)

for col in cat_cols:
    all_data_fe[col] = LabelEncoder().fit_transform(all_data_fe[col].astype(str))

train_fe_enc = all_data_fe.iloc[:n_train].copy()
test_fe_enc  = all_data_fe.iloc[n_train:].copy()

feature_cols_fe = [c for c in train_fe_enc.columns if c != ID_COL]
X_fe      = train_fe_enc[feature_cols_fe].values
X_test_fe = test_fe_enc[feature_cols_fe].values

print(f'Features after engineering: {len(feature_cols_fe)} (was {len(feature_cols)})')
print('New features:', [c for c in feature_cols_fe if c not in feature_cols])

In [ ]:
# --- LightGBM with engineered features ---
print('=== LightGBM + Feature Engineering ===')
lgbm_fe_oof, lgbm_fe_test, lgbm_fe_scores, lgbm_fe_oof_score = cv_with_oof(
    make_lgbm, X_fe, y, X_test_fe, skf, 'LightGBM+FE'
)

---
## 8. Hyperparameter Tuning with Optuna

Tune LightGBM (our best model so far) using Bayesian optimization.

In [ ]:
def objective_lgbm(trial):
    params = {
        'objective': 'multiclass',
        'num_class': 3,
        'metric': 'multi_logloss',
        'verbosity': -1,
        'n_jobs': -1,
        'random_state': SEED,
        'class_weight': 'balanced',
        'n_estimators': trial.suggest_int('n_estimators', 300, 2000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 200),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 0.5),
    }

    # Use 3-fold CV for speed during tuning
    skf3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    scores = []
    for tr_idx, val_idx in skf3.split(X_fe, y):
        model = lgb.LGBMClassifier(**params)
        model.fit(X_fe[tr_idx], y[tr_idx])
        preds = model.predict(X_fe[val_idx])
        scores.append(balanced_accuracy_score(y[val_idx], preds))

    return np.mean(scores)


# Run Optuna study
# Note: n_trials=50 is a good starting point; increase for better results if you have time
N_OPTUNA_TRIALS = 50

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective_lgbm, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)

print(f'\nBest OOF Balanced Accuracy (3-fold, tuning): {study.best_value:.4f}')
print('Best params:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

In [ ]:
# --- Plot Optuna optimization history ---
optuna.visualization.matplotlib.plot_optimization_history(study)
plt.title('Optuna Optimization History')
plt.tight_layout()
plt.show()

In [ ]:
# --- Retrain best LightGBM with tuned params using full 5-fold CV ---
print('=== Tuned LightGBM (5-fold CV) ===')

BEST_PARAMS = {
    'objective': 'multiclass',
    'num_class': 3,
    'metric': 'multi_logloss',
    'verbosity': -1,
    'n_jobs': -1,
    'random_state': SEED,
    'class_weight': 'balanced',
    **study.best_params
}

def make_tuned_lgbm():
    return lgb.LGBMClassifier(**BEST_PARAMS)

tuned_oof, tuned_test, tuned_scores, tuned_oof_score = cv_with_oof(
    make_tuned_lgbm, X_fe, y, X_test_fe, skf, 'Tuned LightGBM'
)

---
## 9. Ensembling

Blend predictions from LightGBM, XGBoost, and CatBoost using their OOF probabilities.

In [ ]:
# Also run XGBoost and CatBoost on engineered features
print('=== XGBoost + Feature Engineering ===')
xgb_fe_oof, xgb_fe_test, xgb_fe_scores, xgb_fe_oof_score = cv_with_oof(
    make_xgb, X_fe, y, X_test_fe, skf, 'XGBoost+FE'
)

print('\n=== CatBoost + Feature Engineering ===')
cat_fe_indices = [feature_cols_fe.index(c) for c in cat_cols if c in feature_cols_fe]

def make_cbc_fe():
    return CatBoostClassifier(**{**CBC_PARAMS, 'cat_features': cat_fe_indices})

cbc_fe_oof, cbc_fe_test, cbc_fe_scores, cbc_fe_oof_score = cv_with_oof(
    make_cbc_fe, X_fe, y, X_test_fe, skf, 'CatBoost+FE'
)

In [ ]:
# --- Simple average ensemble ---
ensemble_oof_avg  = (tuned_oof + xgb_fe_oof + cbc_fe_oof) / 3
ensemble_test_avg = (tuned_test + xgb_fe_test + cbc_fe_test) / 3

ensemble_oof_score = balanced_accuracy_score(y, np.argmax(ensemble_oof_avg, axis=1))
print(f'Simple Average Ensemble OOF Balanced Accuracy: {ensemble_oof_score:.4f}')

In [ ]:
# --- Weighted ensemble (weight by OOF score) ---
w_lgbm = tuned_oof_score
w_xgb  = xgb_fe_oof_score
w_cbc  = cbc_fe_oof_score
total_w = w_lgbm + w_xgb + w_cbc

ensemble_oof_wt  = (w_lgbm * tuned_oof  + w_xgb * xgb_fe_oof  + w_cbc * cbc_fe_oof)  / total_w
ensemble_test_wt = (w_lgbm * tuned_test + w_xgb * xgb_fe_test + w_cbc * cbc_fe_test) / total_w

ensemble_wt_score = balanced_accuracy_score(y, np.argmax(ensemble_oof_wt, axis=1))
print(f'Weighted Ensemble OOF Balanced Accuracy:        {ensemble_wt_score:.4f}')
print(f'  Weights — LGBM: {w_lgbm/total_w:.3f}, XGB: {w_xgb/total_w:.3f}, CBC: {w_cbc/total_w:.3f}')

# Pick the better ensemble
if ensemble_wt_score >= ensemble_oof_score:
    final_test_preds = ensemble_test_wt
    best_ensemble = 'Weighted'
else:
    final_test_preds = ensemble_test_avg
    best_ensemble = 'Simple Average'

print(f'\n→ Using {best_ensemble} ensemble for submission')

In [ ]:
# --- Final Model Comparison Summary ---
summary = pd.DataFrame({
    'Model': ['LightGBM (baseline)', 'XGBoost (baseline)', 'CatBoost (baseline)',
              'LightGBM+FE', 'XGBoost+FE', 'CatBoost+FE',
              'Tuned LightGBM+FE', 'Simple Avg Ensemble', 'Weighted Ensemble'],
    'OOF_Balanced_Accuracy': [
        lgbm_oof_score, xgb_oof_score, cbc_oof_score,
        lgbm_fe_oof_score, xgb_fe_oof_score, cbc_fe_oof_score,
        tuned_oof_score, ensemble_oof_score, ensemble_wt_score
    ]
}).sort_values('OOF_Balanced_Accuracy', ascending=False)

print(summary.to_string(index=False))

plt.figure(figsize=(10, 5))
sns.barplot(data=summary, x='OOF_Balanced_Accuracy', y='Model', palette='viridis')
plt.title('Model Comparison — OOF Balanced Accuracy', fontsize=13)
plt.xlabel('OOF Balanced Accuracy')
plt.xlim(summary['OOF_Balanced_Accuracy'].min() - 0.02, 1.0)
plt.tight_layout()
plt.show()

---
## 10. Final Submission

Generate the submission file and **take a screenshot of the leaderboard after uploading!**

In [ ]:
# Convert predictions back to class labels
final_pred_classes = np.argmax(final_test_preds, axis=1)
final_pred_labels  = [inv_label_map[c] for c in final_pred_classes]

submission = pd.DataFrame({
    'id': test['id'],
    'Irrigation_Need': final_pred_labels
})

print('Submission class distribution:')
print(submission['Irrigation_Need'].value_counts())
print()
print(submission.head(10))

In [ ]:
# Save submission
SUBMISSION_PATH = 'submission.csv'
submission.to_csv(SUBMISSION_PATH, index=False)
print(f'Submission saved to: {SUBMISSION_PATH}')
print('Upload to: https://www.kaggle.com/competitions/playground-series-s6e4/submit')
print()
print('=== REMINDER: Take a screenshot if you hit top 10! ===')

In [ ]:
# Verify format matches sample submission
print('Sample submission format:')
print(sub.head())
print()
print('Our submission format:')
print(submission.head())
print()
print('Shapes match:', submission.shape == sub.shape)
print('Columns match:', list(submission.columns) == list(sub.columns))
print('Valid labels:', set(submission['Irrigation_Need']).issubset({'Low', 'Medium', 'High'}))

---
## Appendix: Tips for Improvement

If you want to push higher on the leaderboard, try these:

1. **More Optuna trials** — increase `N_OPTUNA_TRIALS` to 100–200  
2. **Early stopping** — use `callbacks` with a validation set to avoid overfitting  
3. **Stacking** — train a meta-model on OOF predictions (use `LogisticRegression` as meta-learner)  
4. **Use the original dataset** — Kaggle mentions the original Irrigation Prediction dataset; adding it as extra training data can help  
5. **Pseudo-labeling** — predict on test set, add high-confidence predictions to training data  
6. **Target encoding** — encode categorical features using the mean of the target (use `category_encoders` library)  
7. **Neural network** — try a simple `tabnet` or `pytorch` MLP as an additional ensemble member  
8. **Submit early** — every submission counts; get on the board ASAP  